<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_05_03_XGBoost_one2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_05_03 - ONE2ONE - XGBoost**

## **Introducción**

**Idea básica**

**XGBoost (Extreme Gradient Boosting)** es un modelo de ensamble basado en **árboles de decisión construidos secuencialmente**.

A diferencia de Random Forest (paralelo), XGBoost:

* entrena árboles **uno detrás de otro**
* cada árbol intenta **corregir los errores del anterior**

Formalmente:

$$
\hat{y} = \sum_{m=1}^{M} f_m(x)
$$

donde cada $f_m$ es un árbol.

---

**Cómo funciona**

* Se minimiza una función de pérdida (MSE en regresión)
* Se usa **gradient boosting**
* Cada nuevo árbol aprende sobre:

  * residuos del modelo actual
  * gradiente de la pérdida

---

**Propiedades clave**

* Captura:

  * no linealidades
  * interacciones complejas
* Regularización incorporada:

  * `lambda` (L2)
  * `alpha` (L1)
* Muy eficiente (optimizado en C++)

---

**Por qué es clave en tu proyecto**

* Ridge → no encontró señal
* Random Forest → no secuencial

**XGBoost es el primer modelo:**

* **no lineal**
* **secuencial**
* con **alto poder predictivo en tabular**

Es probablemente tu modelo más importante en esta etapa.

# **Bloque común**

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de variables X e y, y scalers**

In [3]:
from pathlib import Path
import os
import pandas as pd
import joblib


XY_DELTA_DIR = DRIVE_DIR / Path(
    os.environ.get("XY_DELTA_DIR", "data/splits/")
)

XY_DELTA_DIR_SCALED = DRIVE_DIR / Path(
    os.environ.get("XY_DELTA_DIR_SCALED", "data/scaled/")
)

SCALERS_DIR = DRIVE_DIR / Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

TARGETS = ["delta_60", "delta_90"]
SPLITS = ["train", "valid", "test"]


def load_mnq_tabular_split(
    target: str,
    split: str,
    scaled: bool = False,
    return_scaler: bool = False,
):
    if target not in TARGETS:
        raise ValueError(f"target inválido: {target}. Esperados: {TARGETS}")

    if split not in SPLITS:
        raise ValueError(f"split inválido: {split}. Esperados: {SPLITS}")

    x_path = (
        XY_DELTA_DIR_SCALED / f"mnq_{target}_X_{split}_scaled.parquet"
        if scaled
        else XY_DELTA_DIR / f"mnq_{target}_X_{split}.parquet"
    )
    y_path = XY_DELTA_DIR / f"mnq_{target}_y_{split}.parquet"

    if not x_path.exists():
        raise FileNotFoundError(f"No existe X: {x_path}")
    if not y_path.exists():
        raise FileNotFoundError(f"No existe y: {y_path}")

    X = pd.read_parquet(x_path)
    y = pd.read_parquet(y_path)

    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]

    if not return_scaler:
        return X, y

    scaler = None
    if scaled:
        scaler_path = SCALERS_DIR / f"scaler_{target}.pkl"
        if scaler_path.exists():
            scaler = joblib.load(scaler_path)

    return X, y, scaler

In [4]:
#Sin escalado
SIN_ESCALADO = '''
X_train, y_train = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=False,
)
'''

In [5]:
#Escalado
ESCALADO = '''
X_train, y_train = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=True,
)
'''


ESCALADO_ESCALADOR = '''
X_train, y_train, scaler = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=True,
    load_scaler=True,
)
'''

In [6]:
import pandas as pd
import numpy as np


def validate_tabular_dataset(
    X: pd.DataFrame,
    y: pd.Series | pd.DataFrame,
    *,
    name: str = "",
    check_index_alignment: bool = True,
    check_sorted: bool = True,
    date_col: str | None = None,
    verbose: bool = True,
):
    """
    Valida consistencia de un dataset tabular (X, y).

    Checks:
    - shapes
    - NaNs / inf
    - alineación de índices
    - orden temporal (opcional)
    - duplicados

    Retorna
    -------
    dict con flags de validación
    """

    report = {}

    # -------- Convertir y --------
    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]

    # -------- Shapes --------
    report["n_samples_X"] = X.shape[0]
    report["n_samples_y"] = y.shape[0]
    report["n_features"] = X.shape[1]
    report["shape_match"] = X.shape[0] == y.shape[0]

    # -------- NaNs / inf --------
    report["X_has_nan"] = X.isna().any().any()
    report["y_has_nan"] = y.isna().any()

    report["X_has_inf"] = np.isinf(X.select_dtypes(include=[np.number])).any().any()
    report["y_has_inf"] = np.isinf(y).any()

    # -------- Índices --------
    if check_index_alignment:
        report["index_equal"] = X.index.equals(y.index)
    else:
        report["index_equal"] = None

    # -------- Orden temporal --------
    if check_sorted:
        if date_col and date_col in X.columns:
            report["sorted_by_date"] = X[date_col].is_monotonic_increasing
        else:
            report["sorted_by_index"] = X.index.is_monotonic_increasing
    else:
        report["sorted"] = None

    # -------- Duplicados --------
    report["duplicate_index"] = X.index.duplicated().any()

    # -------- Print --------
    if verbose:
        print(f"\n=== VALIDATION: {name} ===")
        for k, v in report.items():
            print(f"{k}: {v}")

        if not report["shape_match"]:
            print("⚠️ ERROR: X e y no tienen mismo número de filas")

        if report["X_has_nan"] or report["y_has_nan"]:
            print("⚠️ WARNING: Hay NaNs")

        if report["X_has_inf"] or report["y_has_inf"]:
            print("⚠️ WARNING: Hay valores infinitos")

        if check_index_alignment and not report["index_equal"]:
            print("⚠️ WARNING: Índices no alineados")

    return report

## **4. Reproducibilidad**

In [7]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [8]:
from pathlib import Path
import os
import sys
import importlib

DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

from metrics.one2one_metrics import evaluate_regression_predictions, print_metrics

print("OK - imports metrics.*")

import metrics.one2one_metrics as m

#print(m.__doc__)
#print(m.evaluate_regression_predictions.__doc__)


OK - imports metrics.*


## **6. Métricas Machine Learning**

In [9]:
import pandas as pd


def evaluate_model_on_split(
    model,
    X,
    y,
    *,
    split_name: str,
    model_name: str,
    target_name: str,
):
    """
    Evalúa un modelo sobre un split dado y devuelve:
    - y_pred
    - dict de métricas
    """
    y_pred = model.predict(X)

    metrics = evaluate_regression_predictions(
        y_true=y,
        y_pred=y_pred,
        split_name=split_name,
        model_name=model_name,
        target_name=target_name,
    )

    return y_pred, metrics


def metrics_to_df(metrics: dict) -> pd.DataFrame:
    """
    Convierte un dict de métricas en una fila de DataFrame.
    Versión final sin redundancias (ni window_size ni horizon).
    """
    row = {
        "model": metrics.get("model"),
        "split": metrics.get("split"),
        "target": metrics.get("target"),
        "n_samples": metrics.get("n_samples"),
        "mae": metrics.get("mae"),
        "rmse": metrics.get("rmse"),
        "r2": metrics.get("r2"),
        "directional_accuracy": metrics.get("directional_accuracy"),
    }

    return pd.DataFrame([row])

def evaluate_model_on_bundle(
    model,
    bundle: dict,
    *,
    model_name: str,
    target_name: str,
    window_size: int | None = None,
    horizon: int | None = None,
    splits: tuple[str, ...] = ("valid", "test"),
):
    """
    Evalúa un modelo en varios splits de un bundle.

    Estructura esperada de bundle:
    bundle = {
        "train": {"X": ..., "y": ...},
        "valid": {"X": ..., "y": ...},
        "test":  {"X": ..., "y": ...},
    }

    Retorna
    -------
    predictions : dict
        Predicciones por split.
    metrics_dict : dict
        Métricas por split.
    metrics_df : pd.DataFrame
        Tabla consolidada.
    """
    predictions = {}
    metrics_dict = {}
    frames = []

    for split in splits:
        X = bundle[split]["X"]
        y = bundle[split]["y"]

        y_pred, metrics = evaluate_model_on_split(
            model=model,
            X=X,
            y=y,
            split_name=split,
            model_name=model_name,
            target_name=target_name,
        )

        predictions[split] = y_pred
        metrics_dict[split] = metrics
        frames.append(
            metrics_to_df(
                metrics,
                window_size=window_size,
                horizon=horizon,
            )
        )

    metrics_df = pd.concat(frames, ignore_index=True)

    return predictions, metrics_dict, metrics_df

## **7. Gestión de dataset de métricas**

In [10]:
def load_one2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/one2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"one2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [11]:
from pathlib import Path
import pandas as pd

def save_one2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/one2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"one2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

# **DEFINICIÓN DE MODELO**

## **8. Definición del modelo — placeholder**

### **8.1. Modelo Ridge Regression (one2one)**

### **8.2. Carga de X e y**

In [12]:
TARGETS = ["delta_60", "delta_90"]

data = {}

for target in TARGETS:
    print(f"\n==============================")
    print(f"CARGANDO DATASET: {target}")
    print(f"==============================")

    # -------- TRAIN --------
    X_train, y_train = load_mnq_tabular_split(
        target=target,
        split="train",
        scaled=False,   # ✔ correcto
    )

    validate_tabular_dataset(
        X_train,
        y_train,
        name=f"train_{target}",
    )

    # -------- VALID --------
    X_valid, y_valid = load_mnq_tabular_split(
        target=target,
        split="valid",
        scaled=False,
    )

    validate_tabular_dataset(
        X_valid,
        y_valid,
        name=f"valid_{target}",
    )

    # -------- TEST --------
    X_test, y_test = load_mnq_tabular_split(
        target=target,
        split="test",
        scaled=False,
    )

    validate_tabular_dataset(
        X_test,
        y_test,
        name=f"test_{target}",
    )

    # -------- Guardar --------
    data[target] = {
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }


CARGANDO DATASET: delta_60

=== VALIDATION: train_delta_60 ===
n_samples_X: 490146
n_samples_y: 490146
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

=== VALIDATION: valid_delta_60 ===
n_samples_X: 104954
n_samples_y: 104954
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

=== VALIDATION: test_delta_60 ===
n_samples_X: 105495
n_samples_y: 105495
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

CARGANDO DATASET: delta_90

=== VALIDATION: train_delta_90 ===
n_samples_X: 490146
n_samples_y: 490146
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_in

In [13]:
X_train_delta_60 = data["delta_60"]["train"]["X"]
y_train_delta_60 = data["delta_60"]["train"]["y"]

X_valid_delta_60 = data["delta_60"]["valid"]["X"]
y_valid_delta_60 = data["delta_60"]["valid"]["y"]

X_test_delta_60  = data["delta_60"]["test"]["X"]
y_test_delta_60  = data["delta_60"]["test"]["y"]

X_train_delta_90 = data["delta_90"]["train"]["X"]
y_train_delta_90 = data["delta_90"]["train"]["y"]

X_valid_delta_90 = data["delta_90"]["valid"]["X"]
y_valid_delta_90 = data["delta_90"]["valid"]["y"]

X_test_delta_90  = data["delta_90"]["test"]["X"]
y_test_delta_90  = data["delta_90"]["test"]["y"]

### **8.3. Entrenamiento**

In [16]:
import torch

def check_gpu_available():
    print("=== GPU CHECK ===")

    cuda_available = torch.cuda.is_available()
    print("torch.cuda.is_available():", cuda_available)

    if cuda_available:
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("No hay GPU disponible")

    return cuda_available

In [17]:
USE_GPU = check_gpu_available()

=== GPU CHECK ===
torch.cuda.is_available(): True
GPU: Tesla T4


In [18]:
from xgboost import XGBRegressor

def check_xgboost_gpu_support():
    print("\n=== XGBOOST GPU CHECK ===")

    try:
        model = XGBRegressor(device="cuda", tree_method="hist")
        params = model.get_xgb_params()

        print("XGBoost acepta parámetro device='cuda'")
        print("Params:", params)

        return True

    except Exception as e:
        print("XGBoost NO soporta GPU en este entorno")
        print("Error:", str(e))
        return False

In [19]:
XGB_GPU_OK = check_xgboost_gpu_support()


=== XGBOOST GPU CHECK ===
XGBoost acepta parámetro device='cuda'
Params: {'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': 'cuda', 'eval_metric': None, 'gamma': None, 'grow_policy': None, 'interaction_constraints': None, 'learning_rate': None, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': None, 'max_leaves': None, 'min_child_weight': None, 'monotone_constraints': None, 'multi_strategy': None, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': None, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': 'hist', 'validate_parameters': None, 'verbosity': None}


In [20]:
USE_GPU = check_gpu_available() and check_xgboost_gpu_support()

print("\nUSE_GPU FINAL:", USE_GPU)

=== GPU CHECK ===
torch.cuda.is_available(): True
GPU: Tesla T4

=== XGBOOST GPU CHECK ===
XGBoost acepta parámetro device='cuda'
Params: {'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': 'cuda', 'eval_metric': None, 'gamma': None, 'grow_policy': None, 'interaction_constraints': None, 'learning_rate': None, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': None, 'max_leaves': None, 'min_child_weight': None, 'monotone_constraints': None, 'multi_strategy': None, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': None, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': 'hist', 'validate_parameters': None, 'verbosity': None}

USE_GPU FINAL: True


In [24]:
import xgboost as xgb
from xgboost import XGBRegressor


def train_evaluate_xgboost_one2one(
    *,
    target_name: str,
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test,
    n_estimators: int = 500,
    learning_rate: float = 0.05,
    max_depth: int = 6,
    min_child_weight: float = 1.0,
    subsample: float = 0.8,
    colsample_bytree: float = 0.8,
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,
    objective: str = "reg:squarederror",
    random_state: int = 42,
    n_jobs: int = -1,
    verbose: int | bool = 50,
    use_gpu: bool = False,
):
    """
    Entrena y evalúa un XGBoost Regressor one-to-one para un target dado.
    Compatible con entornos donde fit() no acepta early_stopping_rounds.
    """

    model_params = dict(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        objective=objective,
        random_state=random_state,
        n_jobs=n_jobs,
        tree_method="hist",
    )

    if use_gpu:
        model_params["device"] = "cuda"

    model = XGBRegressor(**model_params)

    # -------------------------
    # Entrenamiento
    # -------------------------
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_train, y_train), (X_valid, y_valid)],
        verbose=verbose,
    )

    # -------------------------
    # Predicciones
    # -------------------------
    y_pred_valid = model.predict(X_valid)
    y_pred_test = model.predict(X_test)

    # -------------------------
    # Métricas
    # -------------------------
    metrics_valid = evaluate_regression_predictions(
        y_true=y_valid,
        y_pred=y_pred_valid,
        split_name="valid",
        model_name="xgboost",
        target_name=target_name,
    )

    metrics_test = evaluate_regression_predictions(
        y_true=y_test,
        y_pred=y_pred_test,
        split_name="test",
        model_name="xgboost",
        target_name=target_name,
    )

    # -------------------------
    # Historial
    # -------------------------
    evals_result = None
    try:
        evals_result = model.evals_result()
    except Exception:
        pass

    results = {
        "model": model,
        "target": target_name,
        "predictions": {
            "valid": y_pred_valid,
            "test": y_pred_test,
        },
        "metrics": {
            "valid": metrics_valid,
            "test": metrics_test,
        },
        "training": {
            "evals_result": evals_result,
            "device": "cuda" if use_gpu else "cpu",
            "xgboost_version": xgb.__version__,
        },
    }

    return results

In [25]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [26]:
xgb_delta_60 = train_evaluate_xgboost_one2one(
    target_name="delta_60",
    X_train=X_train_delta_60,
    y_train=y_train_delta_60,
    X_valid=X_valid_delta_60,
    y_valid=y_valid_delta_60,
    X_test=X_test_delta_60,
    y_test=y_test_delta_60,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    verbose=50,
    use_gpu=USE_GPU,
)

print_metrics(xgb_delta_60["metrics"]["valid"])
print_metrics(xgb_delta_60["metrics"]["test"])

[0]	validation_0-rmse:52.00197	validation_1-rmse:49.12106
[50]	validation_0-rmse:51.15538	validation_1-rmse:49.07958
[100]	validation_0-rmse:50.77677	validation_1-rmse:49.09377
[150]	validation_0-rmse:50.46572	validation_1-rmse:49.10641
[200]	validation_0-rmse:50.20985	validation_1-rmse:49.12735
[250]	validation_0-rmse:49.98073	validation_1-rmse:49.14618
[300]	validation_0-rmse:49.78444	validation_1-rmse:49.17293
[350]	validation_0-rmse:49.60123	validation_1-rmse:49.18876
[400]	validation_0-rmse:49.42459	validation_1-rmse:49.21225
[450]	validation_0-rmse:49.23763	validation_1-rmse:49.22190
[499]	validation_0-rmse:49.09575	validation_1-rmse:49.23590
=== Regression Metrics ===
model: xgboost
target: delta_60
split: valid
n_samples: 104954
mae: 34.280528
rmse: 49.235905
r2: -0.004984
directional_accuracy: 0.512606
=== Regression Metrics ===
model: xgboost
target: delta_60
split: test
n_samples: 105495
mae: 52.289157
rmse: 82.677901
r2: -0.025929
directional_accuracy: 0.503076


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [18:30:08] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [27]:
xgb_delta_90 = train_evaluate_xgboost_one2one(
    target_name="delta_90",
    X_train=X_train_delta_90,
    y_train=y_train_delta_90,
    X_valid=X_valid_delta_90,
    y_valid=y_valid_delta_90,
    X_test=X_test_delta_90,
    y_test=y_test_delta_90,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    verbose=50,
    use_gpu=USE_GPU,
)

print_metrics(xgb_delta_90["metrics"]["valid"])
print_metrics(xgb_delta_90["metrics"]["test"])

[0]	validation_0-rmse:64.39449	validation_1-rmse:61.47865
[50]	validation_0-rmse:63.37652	validation_1-rmse:61.39724
[100]	validation_0-rmse:62.92661	validation_1-rmse:61.43469
[150]	validation_0-rmse:62.59943	validation_1-rmse:61.45460
[200]	validation_0-rmse:62.29864	validation_1-rmse:61.46346
[250]	validation_0-rmse:62.05295	validation_1-rmse:61.48443
[300]	validation_0-rmse:61.80538	validation_1-rmse:61.51037
[350]	validation_0-rmse:61.60155	validation_1-rmse:61.52518
[400]	validation_0-rmse:61.40335	validation_1-rmse:61.54322
[450]	validation_0-rmse:61.20541	validation_1-rmse:61.56760
[499]	validation_0-rmse:61.02857	validation_1-rmse:61.59098
=== Regression Metrics ===
model: xgboost
target: delta_90
split: valid
n_samples: 104954
mae: 42.936365
rmse: 61.590980
r2: -0.004283
directional_accuracy: 0.520962
=== Regression Metrics ===
model: xgboost
target: delta_90
split: test
n_samples: 105495
mae: 65.923802
rmse: 102.921277
r2: -0.021076
directional_accuracy: 0.500820


### **8.4. Guardado de datasets de métricas**

In [28]:
def build_and_save_one2one_metrics(
    *,
    results_delta_60: dict,
    results_delta_90: dict,
    model_name: str,
):
    """
    Construye y guarda métricas one2one para ambos targets.

    Parámetros
    ----------
    results_delta_60 : dict
    results_delta_90 : dict
    model_name : str
        Nombre del modelo (ej: 'ridge', 'random_forest')
    """

    # =========================================================
    # 1. DELTA 60
    # =========================================================
    df_valid_60 = metrics_to_df(
        results_delta_60["metrics"]["valid"]
    )

    df_test_60 = metrics_to_df(
        results_delta_60["metrics"]["test"]
    )

    df_60 = pd.concat([df_valid_60, df_test_60], ignore_index=True)

    # =========================================================
    # 2. DELTA 90
    # =========================================================
    df_valid_90 = metrics_to_df(
        results_delta_90["metrics"]["valid"]
    )

    df_test_90 = metrics_to_df(
        results_delta_90["metrics"]["test"]
    )

    df_90 = pd.concat([df_valid_90, df_test_90], ignore_index=True)

    # =========================================================
    # 3. CONSOLIDACIÓN
    # =========================================================
    df_all = pd.concat([df_60, df_90], ignore_index=True)

    # =========================================================
    # 4. GUARDADO
    # =========================================================
    save_one2one_metrics(
        df_all,
        name=f"{model_name}_all",
    )

    return df_all

In [30]:
df_xgb = build_and_save_one2one_metrics(
    results_delta_60=xgb_delta_60,
    results_delta_90=xgb_delta_90,
    model_name="xgboost",
)

df_xgb

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/one2one_metrics/one2one_xgboost_all_metrics.parquet


,model,split,target,n_samples,mae,rmse,r2,directional_accuracy
0,xgboost,valid,delta_60,104954,34.280528,49.235905,-0.004984,0.512606
1,xgboost,test,delta_60,105495,52.289157,82.677901,-0.025929,0.503076
2,xgboost,valid,delta_90,104954,42.936365,61.590980,-0.004283,0.520962
3,xgboost,test,delta_90,105495,65.923802,102.921277,-0.021076,0.500820
